# Análisis de Fondo Individual

Notebook para explorar y depurar datos de un fondo concreto.

**Secciones:**
1. Setup
2. Configuración del fondo (cambia aquí el ISIN)
3. NAV actual — comparación de proveedores
4. Histórico de NAV — comparación de proveedores
5. Info general del fondo
6. Sectores
7. Regiones / Países
8. Holdings
9. Diagnóstico completo

## 1. Setup

In [2]:
import sys
import warnings
from pathlib import Path

import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", "{:,.4f}".format)

BACKEND_DIR = Path(".").resolve().parent
if str(BACKEND_DIR) not in sys.path:
    sys.path.insert(0, str(BACKEND_DIR))

from app.client import PortfolioClient
from app.services.data_providers import (
    CompositeProvider, FMPProvider, MStarProvider, YFinanceProvider
)
from app.services.finect_provider import FinectProvider
from app.services.ft_provider import FTProvider

CACHE_PATH = str(BACKEND_DIR / "data" / "cache")
print(f"Backend dir: {BACKEND_DIR}")

Backend dir: C:\Users\jaguirrepeman\OneDrive - Deloitte (O365D)\Documents\DS\Finance\backend


## 2. Configuración del fondo

**Cambia `ISIN` para analizar cualquier fondo.**

In [ ]:
ISIN = "IE00BYX5MX67"   # ← cambia aquí
YEARS = 5

print(f"ISIN analizado: {ISIN}")
print(f"Histórico:      {YEARS} años")

ISIN analizado: IE00BYX5MX67
Histórico:      5 años


: 

## 3. NAV actual — comparación de proveedores

Mostramos el NAV más reciente que devuelve cada proveedor y su fecha.

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime

_providers = {
    "YFinance":  YFinanceProvider(),
    "MorningStr": MStarProvider(cache_path=CACHE_PATH),
    "Finect":    FinectProvider(),
    "FT":        FTProvider(),
}

fmp = FMPProvider()
if fmp.available:
    _providers["FMP"] = fmp

def _probe_nav(name, prov):
    nav, nav_date, hist_rows, hist_last = None, None, 0, None
    try:
        nav = prov.get_nav(ISIN)
        nav_date = prov.get_nav_date(ISIN)
    except Exception as e:
        nav_date = f"ERROR: {e}"
    try:
        hist = prov.get_nav_history(ISIN, years=YEARS)
        if hist is not None and not hist.empty:
            hist_rows = len(hist)
            hist_last = str(pd.to_datetime(hist["date"]).max().date())
    except Exception:
        pass
    return {"Proveedor": name, "NAV": nav, "Fecha_NAV": nav_date,
            "Registros_Hist": hist_rows, "Último_Dato_Hist": hist_last}

results = []
with ThreadPoolExecutor(max_workers=len(_providers)) as ex:
    futures = {ex.submit(_probe_nav, n, p): n for n, p in _providers.items()}
    for f in as_completed(futures):
        results.append(f.result())

df_probe = pd.DataFrame(results).sort_values("Último_Dato_Hist", ascending=False, na_position="last")
display(df_probe.reset_index(drop=True))

Cache manager initialized with base path: C:\Users\jaguirrepeman\OneDrive - Deloitte (O365D)\Documents\DS\Finance\backend\data\cache

--- Procesando: IE00BYX5MX67 (light)---


$FEP7.MU: possibly delisted; no price data found  (period=5d)


✓ Datos obtenidos para IE00BYX5MX67: 1 días
Cache manager initialized with base path: C:\Users\jaguirrepeman\OneDrive - Deloitte (O365D)\Documents\DS\Finance\backend\data\cache
  ✓ Usando datos en caché para IE00BYX5MX67 (light)
Cache manager initialized with base path: C:\Users\jaguirrepeman\OneDrive - Deloitte (O365D)\Documents\DS\Finance\backend\data\cache
  ✓ Usando datos en caché para IE00BYX5MX67 (light)


,Proveedor,NAV,Fecha_NAV,Registros_Hist,Último_Dato_Hist
0,YFinance,NaN,NaN,1,2025-06-11
1,MorningStr,12.7730,2025-06-11,1,2025-06-11
2,FMP,NaN,NaN,0,NaN
3,FT,15.1400,NaN,0,NaN
4,Finect,15.1437,2026-05-01,0,NaN


: 

In [ ]:
# NAV del CompositeProvider (elige el más reciente automáticamente)
composite = CompositeProvider(cache_path=CACHE_PATH)
nav_composite = composite.get_nav(ISIN)
nav_date_composite = composite.get_nav_date(ISIN)

print(f"CompositeProvider → NAV={nav_composite}  Fecha={nav_date_composite}")

Cache manager initialized with base path: C:\Users\jaguirrepeman\OneDrive - Deloitte (O365D)\Documents\DS\Finance\backend\data\cache
  ✓ Usando datos en caché para IE00BYX5MX67 (light)
Cache manager initialized with base path: C:\Users\jaguirrepeman\OneDrive - Deloitte (O365D)\Documents\DS\Finance\backend\data\cache
  ✓ Usando datos en caché para IE00BYX5MX67 (light)
CompositeProvider → NAV=12.77299976348877  Fecha=2025-06-11


: 

## 4. Histórico de NAV — comparación de proveedores

In [ ]:
fig = go.Figure()

COLORS = ["#00d4aa", "#8b5cf6", "#ff6b6b", "#ffd93d", "#6bcb77"]

for idx, (name, prov) in enumerate(_providers.items()):
    try:
        hist = prov.get_nav_history(ISIN, years=YEARS)
        if hist is not None and not hist.empty:
            hist["date"] = pd.to_datetime(hist["date"])
            last = hist["date"].max().date()
            fig.add_trace(go.Scatter(
                x=hist["date"],
                y=hist["price"],
                name=f"{name} (último: {last})",
                mode="lines",
                line=dict(color=COLORS[idx % len(COLORS)], width=1.5),
            ))
    except Exception as e:
        print(f"{name}: error — {e}")

fig.update_layout(
    title=f"Histórico NAV — {ISIN} ({YEARS}a) por proveedor",
    template="plotly_dark",
    xaxis_title="Fecha",
    yaxis_title="NAV",
    height=500,
    legend=dict(orientation="h", y=-0.2),
    hovermode="x unified",
)
fig.show()

Cache manager initialized with base path: C:\Users\jaguirrepeman\OneDrive - Deloitte (O365D)\Documents\DS\Finance\backend\data\cache
  ✓ Usando datos en caché para IE00BYX5MX67 (light)


: 

In [ ]:
# Historial seleccionado por CompositeProvider (el más reciente)
df_hist_best = composite.get_nav_history(ISIN, years=YEARS)
if not df_hist_best.empty:
    df_hist_best["date"] = pd.to_datetime(df_hist_best["date"])
    print(f"Registros: {len(df_hist_best)} | "
          f"Desde: {df_hist_best['date'].min().date()} | "
          f"Hasta: {df_hist_best['date'].max().date()}")
    display(df_hist_best.tail(10))
else:
    print("Sin historial disponible.")

Cache manager initialized with base path: C:\Users\jaguirrepeman\OneDrive - Deloitte (O365D)\Documents\DS\Finance\backend\data\cache
  ✓ Usando datos en caché para IE00BYX5MX67 (light)
Registros: 1 | Desde: 2025-06-11 | Hasta: 2025-06-11


,date,price
0,2025-06-11,12.7730


: 

## 5. Info general del fondo

In [ ]:
info = composite.get_fund_info(ISIN)
pd.DataFrame([info]).T.rename(columns={0: "Valor"})

,Valor
isin,IE00BYX5MX67
source,Finect
name,Fidelity S&P 500 Index Fund
management_company,"FIL Investment Management (Luxembourg) S.A., I..."
category,RV USA Cap. Grande Blend
description,El objetivo de inversión es proporcionar a los...
strategy,"El Fidelity S&P 500 Index Fund, con ISIN IE00B..."
srri,6
total_net_asset,"1,684,413,494.5000"
rating_morningstar,4


: 

In [ ]:
# Info de cada proveedor individualmente
for name, prov in _providers.items():
    try:
        info_p = prov.get_fund_info(ISIN)
        if info_p:
            print(f"\n── {name} ──")
            for k, v in info_p.items():
                print(f"  {k}: {v}")
    except Exception as e:
        print(f"{name}: ERROR — {e}")


── YFinance ──
  name: Fidelity Ucits II Icav - Fidelity S&P 500 Index Fund
  currency: EUR
  source: YahooFinance
Cache manager initialized with base path: C:\Users\jaguirrepeman\OneDrive - Deloitte (O365D)\Documents\DS\Finance\backend\data\cache

--- Procesando: IE00BYX5MX67 (detailed)---
✓ Datos obtenidos desde caché para IE00BYX5MX67
Obteniendo datos completos de Morningstar para IE00BYX5MX67...
Recopilando datos completos para IE00BYX5MX67...
❌ Error general obteniendo datos para IE00BYX5MX67: Error 403
            for the api https://global.morningstar.com/api/v1/fr/stores/data-points/fields. Message : Forbidden.
  [Info] Usando datos históricos de Yahoo para IE00BYX5MX67
  ⚠️ Datos con error, no se cachean para IE00BYX5MX67 (detailed)
Cache manager initialized with base path: C:\Users\jaguirrepeman\OneDrive - Deloitte (O365D)\Documents\DS\Finance\backend\data\cache

--- Procesando: IE00BYX5MX67 (detailed)---
✓ Datos obtenidos para IE00BYX5MX67: 1 días
Obteniendo datos completos

: 

## 6. Sectores

In [ ]:
sectors = composite.get_sector_weights(ISIN)

if sectors:
    df_s = pd.DataFrame(sectors.items(), columns=["Sector", "Peso"]).sort_values("Peso", ascending=False)
    
    fig = go.Figure(go.Bar(
        x=df_s["Peso"],
        y=df_s["Sector"],
        orientation="h",
        text=df_s["Peso"].apply(lambda v: f"{v:.1f}%"),
        textposition="outside",
        marker_color="#00d4aa",
    ))
    fig.update_layout(
        title=f"Sectores — {ISIN}",
        template="plotly_dark",
        height=max(300, 35 * len(df_s)),
        margin=dict(l=200),
    )
    fig.show()
    display(df_s)
else:
    print("Sin datos de sectores.")

,Sector,Peso
7,Technology,36.0461
9,Financial Services,11.6505
1,Communication Services,10.7481
2,Consumer Cyclical,10.1525
4,Healthcare,8.4085
5,Industrials,8.2098
3,Consumer Defensive,4.8398
8,Energy,3.3586
10,Utilities,2.3175
6,Real Estate,1.8981


: 

## 7. Regiones / Países

In [ ]:
regions = composite.get_country_weights(ISIN)

if regions:
    df_r = pd.DataFrame(regions.items(), columns=["Región", "Peso"]).sort_values("Peso", ascending=False)
    
    fig = go.Figure(go.Bar(
        x=df_r["Peso"],
        y=df_r["Región"],
        orientation="h",
        text=df_r["Peso"].apply(lambda v: f"{v:.1f}%"),
        textposition="outside",
        marker_color="#8b5cf6",
    ))
    fig.update_layout(
        title=f"Regiones — {ISIN}",
        template="plotly_dark",
        height=max(300, 35 * len(df_r)),
        margin=dict(l=200),
    )
    fig.show()
    display(df_r)
else:
    print("Sin datos de regiones.")

,Región,Peso
0,Americas,98.9300
1,United States,98.9300
5,Greater Europe,0.4200
6,Europe - ex Euro,0.2600
7,Eurozone,0.1300
8,United Kingdom,0.0300
4,Greater Asia,0.0000
2,Canada,0.0000
3,Latin America,0.0000


: 

## 8. Holdings

In [ ]:
df_hold = composite.get_holdings(ISIN)

if not df_hold.empty:
    df_hold_plot = df_hold.sort_values("weight", ascending=True).tail(15)
    
    fig = go.Figure(go.Bar(
        x=df_hold_plot["weight"],
        y=df_hold_plot["name"],
        orientation="h",
        text=df_hold_plot["weight"].apply(lambda v: f"{v:.2f}%"),
        textposition="outside",
        marker_color="#ffd93d",
    ))
    fig.update_layout(
        title=f"Top Holdings — {ISIN}",
        template="plotly_dark",
        height=max(350, 35 * len(df_hold_plot)),
        margin=dict(l=250),
    )
    fig.show()
    display(df_hold)
else:
    print("Sin datos de holdings.")

,name,ticker,weight,market_value
0,NVIDIA Corp,,8.1500,"180,951,641.0000"
1,Apple Inc,,6.4100,"142,285,628.0000"
2,Microsoft Corp,,5.0800,"112,735,336.0000"
3,Amazon.com Inc,,4.1500,"92,212,763.0000"
4,Broadcom Inc,,3.2300,"71,667,543.0000"
5,Alphabet Inc Class A,,3.2300,"71,783,292.0000"
6,Alphabet Inc Class C,,2.5700,"57,148,270.0000"
7,Meta Platforms Inc Class A,,2.3800,"52,792,071.0000"
8,Tesla Inc,,1.7000,"37,847,125.0000"
9,Berkshire Hathaway Inc Class B,,1.3900,"30,762,987.0000"


: 

## 9. Diagnóstico completo

In [ ]:
# Resumen de cobertura de cada proveedor para este fondo
diag = []
for name, prov in {**_providers, "Composite": composite}.items():
    row = {"Proveedor": name}
    try:
        nav = prov.get_nav(ISIN)
        row["NAV"] = f"{nav:.4f}" if nav else "—"
    except Exception:
        row["NAV"] = "ERROR"
    try:
        row["Fecha_NAV"] = prov.get_nav_date(ISIN) or "—"
    except Exception:
        row["Fecha_NAV"] = "ERROR"
    try:
        h = prov.get_nav_history(ISIN, years=YEARS)
        row["Hist_Registros"] = len(h) if h is not None else 0
        if h is not None and not h.empty:
            row["Hist_Desde"] = str(pd.to_datetime(h["date"]).min().date())
            row["Hist_Hasta"] = str(pd.to_datetime(h["date"]).max().date())
        else:
            row["Hist_Desde"] = "—"
            row["Hist_Hasta"] = "—"
    except Exception as e:
        row["Hist_Registros"] = "ERROR"
        row["Hist_Desde"] = str(e)[:40]
        row["Hist_Hasta"] = "—"
    try:
        info_p = prov.get_fund_info(ISIN)
        row["Nombre"] = info_p.get("name", "—")[:50] if info_p else "—"
    except Exception:
        row["Nombre"] = "ERROR"
    try:
        sw = prov.get_sector_weights(ISIN)
        row["Sectores"] = len(sw) if sw else 0
    except Exception:
        row["Sectores"] = "ERROR"
    try:
        cw = prov.get_country_weights(ISIN)
        row["Regiones"] = len(cw) if cw else 0
    except Exception:
        row["Regiones"] = "ERROR"
    try:
        hd = prov.get_holdings(ISIN)
        row["Holdings"] = len(hd) if hd is not None else 0
    except Exception:
        row["Holdings"] = "ERROR"
    diag.append(row)

pd.DataFrame(diag).set_index("Proveedor")

$FEP7.MU: possibly delisted; no price data found  (period=5d)


Cache manager initialized with base path: C:\Users\jaguirrepeman\OneDrive - Deloitte (O365D)\Documents\DS\Finance\backend\data\cache

--- Procesando: IE00BYX5MX67 (light)---
✓ Datos obtenidos para IE00BYX5MX67: 1 días
Cache manager initialized with base path: C:\Users\jaguirrepeman\OneDrive - Deloitte (O365D)\Documents\DS\Finance\backend\data\cache
  ✓ Usando datos en caché para IE00BYX5MX67 (light)
Cache manager initialized with base path: C:\Users\jaguirrepeman\OneDrive - Deloitte (O365D)\Documents\DS\Finance\backend\data\cache
  ✓ Usando datos en caché para IE00BYX5MX67 (light)
Cache manager initialized with base path: C:\Users\jaguirrepeman\OneDrive - Deloitte (O365D)\Documents\DS\Finance\backend\data\cache

--- Procesando: IE00BYX5MX67 (detailed)---
✓ Datos obtenidos desde caché para IE00BYX5MX67
Obteniendo datos completos de Morningstar para IE00BYX5MX67...
Recopilando datos completos para IE00BYX5MX67...
❌ Error general obteniendo datos para IE00BYX5MX67: Error 403
            f

,NAV,Fecha_NAV,Hist_Registros,Hist_Desde,Hist_Hasta,Nombre,Sectores,Regiones,Holdings
Proveedor,,,,,,,,,
YFinance,—,—,1,2025-06-11,2025-06-11,Fidelity Ucits II Icav - Fidelity S&P 500 Inde...,0,0,0
MorningStr,12.7730,2025-06-11,1,2025-06-11,2025-06-11,IE00BYX5MX67,0,0,0
Finect,—,—,0,—,—,Fidelity S&P 500 Index Fund,11,7,10
FT,15.1400,—,0,—,—,Fidelity S&P 500 Index Fund EUR P Acc,10,9,10
FMP,—,—,0,—,—,—,0,0,0
Composite,12.7730,2025-06-11,1,2025-06-11,2025-06-11,Fidelity S&P 500 Index Fund,11,9,10


: 

## 10. Características del fondo (estrellas, rating, TER)

In [ ]:
# PortfolioClient (toda la cartera)
ORDERS_PATH = str(BACKEND_DIR / "data" / "Órdenes 1238478.tsv")
client = PortfolioClient(ORDERS_PATH, cache_path=CACHE_PATH)

# Tabla 1: características estáticas (estrellas, rating de riesgo, TER)
df_char = client.fund_characteristics()
print("── Características de fondos ──")
display(df_char)

Cache manager initialized with base path: C:\Users\jaguirrepeman\OneDrive - Deloitte (O365D)\Documents\DS\Finance\backend\data\cache
  ✓ Usando datos en caché para ES0146309002 (light)


AttributeError: 'PortfolioClient' object has no attribute 'fund_characteristics'

: 

## 11. Métricas de rendimiento por período (1Y / 3Y / 5Y / 10Y)

In [ ]:
# Tabla 2: métricas de rendimiento/riesgo por período temporal
df_metrics = client.fund_metrics()
print("── Métricas por período ──")
display(df_metrics)

# Debug: campos de stats disponibles para el ISIN del notebook
print(f"\n── Campos de stats disponibles en Finect para {ISIN} ──")
_info_debug = composite.get_fund_info(ISIN)
_stat_keys = {k: v for k, v in _info_debug.items()
              if any(s in k for s in ("return", "sharpe", "alpha", "beta",
                                      "standard", "drawdown", "volatil"))}
for k, v in sorted(_stat_keys.items()):
    print(f"  {k}: {v}")

: 

: 